# Wildfire Exploratory Data Analysis

## Objectives

* Explore wildfire trends across Europe, 1980-2024
* Focus analysis on Spain, Portugal, France, and Greece
* Validate project hypothesis around Southern European wildfire trends
* Produce visualisations for the dashboard (min. 2 plot types)

## Inputs

* inputs/processed/wildfires_long_format.csv

## Outputs

* Charts/insights to be reused in the Streamlit dashboard

## Additional Comments

* Focus countries chosen because they have complete EFFIS reporting since 1980
  and are central to the 2025-2026 wildfire crisis motivating this project

---

In [1]:
import pandas as pd

# Change working directory

* We are assuming you will store the notebooks in a subfolder, therefore when running the notebook in the editor, you will need to change the working directory

We need to change the working directory from its current folder to its parent folder
* We access the current directory with os.getcwd()

In [2]:
import os
current_dir = os.getcwd()
current_dir

'/Users/tildeholmqvist/Documents/VS_Code_Tilde/DA_project_3/jupyter_notebooks'

We want to make the parent of the current directory the new current directory
* os.path.dirname() gets the parent directory
* os.chdir() defines the new current directory

In [3]:
os.chdir(os.path.dirname(current_dir))
print("You set a new current directory")

You set a new current directory


Confirm the new current directory

In [4]:
current_dir = os.getcwd()
current_dir

'/Users/tildeholmqvist/Documents/VS_Code_Tilde/DA_project_3'

# Section 1: Load Processed Data

Load the cleaned long-format dataset produced in `01_data_cleaning.ipynb`.

In [5]:
df = pd.read_csv("inputs/processed/wildfires_long_format.csv")

print(df.shape)
df.head()

(922, 5)


,Year,country_iso3,burnt_area_ha,number_of_fires,country_name
0,1980,PRT,44251.0,2349.0,Portugal
1,1981,PRT,89798.0,6730.0,Portugal
2,1982,PRT,39556.0,3626.0,Portugal
3,1983,PRT,47811.0,4539.0,Portugal
4,1984,PRT,52710.0,7356.0,Portugal


---

# Section 2: Focus Countries Overview

Filter the dataset to Spain, Portugal, France, and Greece, and visualise burnt
area trends over time for each.

In [6]:
focus_countries = ['Spain', 'Portugal', 'France', 'Greece']
df_focus = df[df['country_name'].isin(focus_countries)]

print(df_focus.shape)
df_focus.head()

(180, 5)


,Year,country_iso3,burnt_area_ha,number_of_fires,country_name
0,1980,PRT,44251.0,2349.0,Portugal
1,1981,PRT,89798.0,6730.0,Portugal
2,1982,PRT,39556.0,3626.0,Portugal
3,1983,PRT,47811.0,4539.0,Portugal
4,1984,PRT,52710.0,7356.0,Portugal


## Burnt Area Trends Over Time

In [7]:
import plotly.express as px

fig = px.line(df_focus, x='Year', y='burnt_area_ha', color='country_name',
              title='Burnt Area Over Time: Spain, Portugal, France, Greece (1980-2024)',
              labels={'burnt_area_ha': 'Burnt Area (hectares)', 'Year': 'Year', 'country_name': 'Country'})
fig.show()

**In Plain Language:** Each line shows one country's wildfire damage per year, measured
in hectares burnt. Spikes upward mean a particularly bad fire season for that country.
Hover over any point to see the exact year and country.

**Observation:** Spain shows the largest year-to-year swings in the 1980s and 1990s,
with several years exceeding 400,000 hectares burnt, followed by much calmer years.
Portugal recorded the single highest spike in the dataset, coinciding with the
country's most severe wildfire season on record. Greece shows an isolated but
extreme spike around 2007, matching the well-documented Greek wildfire disasters
that year. France remains consistently lower than the other three countries
throughout the entire 45-year period, with no comparable extreme spikes.

---

# Section 3: Reshape data to Long Format

Convert both datasets from wide format (one column per country) to long format
(one row per year-country combination), then merge them into a single tidy dataset.

In [8]:
burnt_area_long = burnt_area.melt(id_vars='Year', var_name='country_iso3', value_name='burnt_area_ha')

burnt_area_long.head()

NameError: name 'burnt_area' is not defined

In [ ]:
print("Total rows:", burnt_area_long.shape[0])
print("Number of unique countries:", burnt_area_long['country_iso3'].nunique())
print("All countries:", sorted(burnt_area_long['country_iso3'].unique()))

Total rows: 1395
Number of unique countries: 31
All countries: ['AUT', 'BGR', 'CHE', 'CYP', 'CZE', 'DEU', 'DZA', 'ESP', 'EST', 'FIN', 'FRA', 'GRC', 'HRV', 'HUN', 'ITA', 'LBN', 'LTU', 'LVA', 'MAR', 'MKD', 'NLD', 'NOR', 'POL', 'PRT', 'ROU', 'SRB', 'SVK', 'SVN', 'SWE', 'TUR', 'UKR']


The reshaped dataset confirms all 31 countries are present, with 45 rows each
(1980-2024) — the `.head()` preview only showed Portugal because the melted rows
follow the original column order, not because other countries were dropped.

In [ ]:
num_fires_long = num_fires.melt(id_vars='Year', var_name='country_iso3', value_name='number_of_fires')

print(num_fires_long.shape)
num_fires_long.head()

(1395, 3)


,Year,country_iso3,number_of_fires
0,1980,PRT,2349.0
1,1981,PRT,6730.0
2,1982,PRT,3626.0
3,1983,PRT,4539.0
4,1984,PRT,7356.0


In [ ]:
print("Total rows:", num_fires_long.shape[0])
print("Number of unique countries:", num_fires_long['country_iso3'].nunique())
print("All countries:", sorted(num_fires_long['country_iso3'].unique()))

Total rows: 1395
Number of unique countries: 31
All countries: ['AUT', 'BGR', 'CHE', 'CYP', 'CZE', 'DEU', 'DZA', 'ESP', 'EST', 'FIN', 'FRA', 'GRC', 'HRV', 'HUN', 'ITA', 'LBN', 'LTU', 'LVA', 'MAR', 'MKD', 'NLD', 'NOR', 'POL', 'PRT', 'ROU', 'SRB', 'SVK', 'SVN', 'SWE', 'TUR', 'UKR']


Same structure and country count as the burnt area dataset, confirming both are
ready to be merged on `Year` and `country_iso3`.

## Merge burnt area and number of fires into a single dataset

Both long-format tables share the same structure (Year, country_iso3), so they can
be merged into one tidy dataset with both indicators as separate columns.

In [ ]:
wildfires_long = pd.merge(burnt_area_long, num_fires_long, on=['Year', 'country_iso3'], how='outer')

print(wildfires_long.shape)
wildfires_long.head()

(1395, 4)


,Year,country_iso3,burnt_area_ha,number_of_fires
0,1980,PRT,44251.0,2349.0
1,1981,PRT,89798.0,6730.0
2,1982,PRT,39556.0,3626.0
3,1983,PRT,47811.0,4539.0
4,1984,PRT,52710.0,7356.0


## Add readable country names

ISO3 codes are precise but not reader-friendly for a non-technical dashboard
audience, so they are mapped to full country names.

In [ ]:
iso3_to_name = {
    'PRT': 'Portugal', 'ESP': 'Spain', 'FRA': 'France', 'ITA': 'Italy', 'GRC': 'Greece',
    'DZA': 'Algeria', 'AUT': 'Austria', 'BGR': 'Bulgaria', 'HRV': 'Croatia', 'CYP': 'Cyprus',
    'CZE': 'Czechia', 'EST': 'Estonia', 'FIN': 'Finland', 'DEU': 'Germany', 'HUN': 'Hungary',
    'LVA': 'Latvia', 'LBN': 'Lebanon', 'LTU': 'Lithuania', 'MAR': 'Morocco', 'NLD': 'Netherlands',
    'MKD': 'North Macedonia', 'NOR': 'Norway', 'POL': 'Poland', 'ROU': 'Romania', 'SRB': 'Serbia',
    'SVK': 'Slovakia', 'SVN': 'Slovenia', 'SWE': 'Sweden', 'CHE': 'Switzerland', 'TUR': 'Turkey',
    'UKR': 'Ukraine'
}

wildfires_long['country_name'] = wildfires_long['country_iso3'].map(iso3_to_name)

unmapped = wildfires_long[wildfires_long['country_name'].isna()]['country_iso3'].unique()
print("Unmapped ISO3 codes (should be empty):", unmapped)

Unmapped ISO3 codes (should be empty): []


## Drop rows with no data at all

Some (year, country) combinations have no data for either indicator, since that
country had not yet joined EFFIS reporting. These rows carry no information and
are removed. Rows with only partial data are kept, since they are still informative.

In [ ]:
before = len(wildfires_long)
wildfires_long = wildfires_long.dropna(subset=['burnt_area_ha', 'number_of_fires'], how='all')
after = len(wildfires_long)

print(f"Dropped {before - after} fully-empty rows. Remaining: {after} rows.")

Dropped 473 fully-empty rows. Remaining: 922 rows.


---

# Section 4: Save Processed Dataset

In [ ]:
wildfires_long.to_csv("inputs/processed/wildfires_long_format.csv", index=False)
print(f"Saved {len(wildfires_long)} rows to inputs/processed/wildfires_long_format.csv")

Saved 922 rows to inputs/processed/wildfires_long_format.csv


# Conclusions and Next Steps

This notebook loaded, inspected, reshaped, and cleaned the raw EFFIS wildfire data
for 31 countries (1980-2024), producing a tidy long-format dataset with 922 rows.

**Key takeaways:**
* Spain, Portugal, France, Italy, and Greece have complete reporting from 1980
  onwards, confirming they are well-suited as the focus countries for this project
* Countries with gaps in early years had not yet joined EFFIS reporting; these
  rows were dropped rather than imputed, to avoid fabricating data

**Output:** `inputs/processed/wildfires_long_format.csv`, ready for exploratory
data analysis.

**Next step:** `02_eda.ipynb` — exploratory analysis of wildfire trends, with a
focus on Spain, Portugal, France, and Greece in the context of the 2025-2026
wildfire crisis.